# Anatomía de una Falla · Parte II — 02 · RUL con incertidumbre (Línea 1)

**Pregunta:** ¿cuántos vuelos le quedan a un motor, y con qué margen de error, controlando especialmente el riesgo de
**sobreestimar** su vida útil?

**Modelos:** CatBoost (baseline, regresión por cuantiles) y Temporal Fusion Transformer (modelo principal), ambos con
intervalos de **Adaptive Conformal Prediction** calibrados en motores que ningún modelo vio.

Los resultados se generan con `python scripts/run_all.py --only catboost tft conformal` y se leen de `outputs/<modo>/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from src.ncmapss import nb
MODE = "full"          # "sample" si corriste el pipeline con --sample

In [ ]:
R = nb.load("results", MODE)
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.DataFrame(R["split_summary"])

## Metodología

1. **Partición por motor** (sin fuga): test = motores de prueba oficiales; los de desarrollo se dividen en
   entrenamiento, validación (early stopping) y calibración (intervalos).
2. **Variables** (sólo hacia atrás): residuos del vuelo, residuo menos la línea base del motor, medias y pendientes
   móviles, número de vuelo, clase y descriptores del vuelo.
3. **Modelos** con salida por cuantiles (5 %, 50 %, 95 %).
4. **Conformal**: se corrigen los cuantiles con los errores en motores de calibración (CQR) y luego se adapta el nivel
   con **ACI** a medida que los motores de la flota fallan y se conoce su RUL real. La cola de **sobreestimación** recibe
   menos tolerancia que la de subestimación.
5. **Métricas**: RMSE, MAE, score asimétrico de NASA, cobertura y ancho de los intervalos y % de sobreestimación.

In [ ]:
pd.DataFrame(R['metrics_table'])

In [ ]:
nb.show('pred_vs_true', R)

In [ ]:
nb.show('coverage_by_rul', R)

In [ ]:
nb.show('aci_trace', R)

In [ ]:
nb.show('engine_examples', R)

In [ ]:
nb.show('error_by_rul', R)

In [ ]:
nb.show('feature_importance', R)

## Lectura de los resultados

In [ ]:
for t in R['discussion']: nb.html('<p>'+t+'</p>')